# A.11 Suffix notation for auxiliary values

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Variables, constraints, objectives and problems have a variety of associated auxiliary values.
For example, variables have bounds and reduced costs, and constraints have dual values and slacks.
Such values are accessed as name .suffix, where name is a simple or subscripted variable, constraint,
objective or problem name, and .suffix is one of the possibilities listed in Tables A-6, A-7,
and A-8.

For a constraint, the . body, . lb, and . ub values correspond to a modified form of the constraint.
If the constraint involves a single inequality, subtract the right-hand side from the left, then
move any constant terms to the right-hand side; if the constraint involves a double inequality, similarly
subtract any constant terms in the middle from all three expressions (left, middle, right). Then
the constraint has the form lb ≤ body ≤ ub, where lb and ub are (possibly infinite) constants.

The following rules determine lower and upper dual values (c . ldual and c . udual) for a
constraint c. The solver returns a single dual value, c . dual, which might apply either to
body ≥ lb or to body ≤ ub. For an equality constraint (lb = ub), AMPL uses the sign of c . dual
to decide. For a minimization problem, c . dual > 0 implies that the same optimum would be
found if the constraint were body ≥ lb, so AMPL sets c . ldual = c . dual and c . udual = 0;
similarly, c . dual < 0 implies that c . ldual = 0 and c . udual = c . dual. For a maximization
problem, the inequalities are reversed.

For inequality constraints (lb < ub), AMPL uses nearness to bound to decide whether
c . ldual or c . udual equals c . dual. If body − lb < ub − body, then c . ldual = c . dual
and c . udual = 0; otherwise, c . ldual = 0 and c . udual = c . dual.

Model declarations may reference any of the suffixed values described in Tables A-6, A-7 and
A-8. This is most often useful in new declarations that are introduced after one model has already
been translated and solved. In particular, the suffixes .val and .dual are provided so that new
constraints can refer to current optimal values of the primal and dual variables and of the objective.

```ampl
.astatus     AMPL status (A.11.2)
.init        current initial guess
.init0       initial initial guess (set by :=, data, or default)
.lb          current lower bound
.lb0         initial lower bound
.lb1         weaker lower bound from presolve
.lb2         stronger lower bound from presolve
.lrc         lower reduced cost (for var >= lb)
.lslack      lower slack (val - lb)
.rc          reduced cost
.relax       ignore integrality restriction if positive
.slack       min(lslack, uslack)
.sstatus     solver status (A.11.2)
.status      status (A.11.2)
.ub          current upper bound
.ub0         initial upper bound
.ub1         weaker upper bound from presolve
.ub2         stronger upper bound from presolve
.urc         upper reduced cost (for var <= ub)
.uslack      upper slack (ub - val)
.val         current value of variable
```

**Table A-6:** Dot suffixes for variables.

For a complementarity constraint, suffix notations like constraint . lb, constraint . body, etc.,
are extended so that constraint . Lsuffix and constraint . Rsuffix correspond to constr 1 .suffix and
constr 2 .suffix, respectively, and complementarity - constraint . slack (or the unadorned name)
stands for a measure of the extent to which the complementarity constraint is satisfied: if constr 1
and constr 2 each involve one inequality, the new measure is

```ampl
min(constr 1 . slack, constr 2 . slack),
```

which is positive if both are satisfied as strict inequalities, 0 if the complementarity constraint is
satisfied exactly, and negative if at least one of constr 1 or constr 2 is violated. For constraints of
the form expr 1 <= expr 2 <= expr 3 complements expr 4 , the .slack value is

```ampl
min(expr 2 -expr 1 , expr 4 ) if expr 1 >= expr 2
min(expr 3 -expr 2 , -expr 4 ) if expr 3 <= expr 2
-abs(expr 4 ) if expr 1 < expr 2 < expr 3
```

so in all cases, the .slack value is 0 if the complementarity constraint holds exactly and is negative
if one of the requisite inequalities is violated.